# Debugging autoreload

In [ ]:
%load_ext autoreload
%autoreload 2

# Load packages

In [ ]:
import pandas as pd
import rootutils
from pytorch_tabular import TabularModel
import pandas as pd
from tqdm import tqdm
import sys
sys.path.append("..")

# Load models

In [ ]:
path_root = str(rootutils.find_root(indicator=".project-root"))

df_imms = pd.read_excel(f"{path_root}/models/InflammatoryMarkers/InflammatoryMarkers.xlsx", index_col='feature')
imms = df_imms.index.values
imms_log = [f"{f}_log" for f in imms]
cpgs = pd.read_excel(f"{path_root}/models/InflammatoryMarkers/CpGs.xlsx", index_col=0).index.values

models_imms = {}
for imm in (pbar := tqdm(imms)):
    pbar.set_description(f"Loading model for {imm}")
    models_imms[imm] = TabularModel.load_model(f"{path_root}/models/InflammatoryMarkers/{imm}")

model_age = TabularModel.load_model(f"{path_root}/models/EpInflammAge")

# Load data example

In [ ]:
data = pd.read_excel(f"{path_root}/data/examples/GSE87571.xlsx", index_col=0)

# Models inference

In [ ]:
# First step: inflammatory markers (logarithmic values)
for imm in (pbar := tqdm(imms)):
    pbar.set_description(f"Inference for {imm}")
    data[f"{imm}_log"] = models_imms[imm].predict(data)
# Second step: age prediction
data['EpInflammAge'] = model_age.predict(data.loc[:, [f"{imm}_log" for imm in imms]])
data['Age Acceleration'] = data['EpInflammAge'] - data['Age']